# Churn Prediction, Risk Profiling, and Lift Chart Evaluation

**Original Question:** How can we predict which customers will churn in the next 30 days, create churn risk profiles, and evaluate the model with a lift chart?

_Exported from PlotStudio_

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


# PlotStudio stub — no-op outside the app
def add_to_workspace(df):
    pass


# Load source data
df_customer_churn = pd.read_csv("data/df_customer_churn.csv")

## Task 1: Ensure the dataset is fully clean, all features are usable, and the analysis population and group definitions are correct for predictive modeling.

**Acceptance Criteria:** TotalCharges is numeric and missing values are handled; the analysis population and churn group sizes are validated; all features are available pre-churn; no target leakage or sample definition errors.

### 1.1: Convert TotalCharges to numeric and handle non-numeric entries.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

df_customer_churn_cleaned = df_customer_churn.copy()
df_customer_churn_cleaned["TotalCharges"] = pd.to_numeric(
    df_customer_churn_cleaned["TotalCharges"], errors="coerce"
)

coerced_missing_count = int(df_customer_churn_cleaned["TotalCharges"].isna().sum())
df_customer_churn_cleaned["TotalCharges"] = df_customer_churn_cleaned[
    "TotalCharges"
].fillna(0.0)
final_row_count = int(len(df_customer_churn_cleaned))
final_dtype = str(df_customer_churn_cleaned["TotalCharges"].dtype)

add_to_workspace(df_customer_churn_cleaned)

print("Validation summary:")
print(f"TotalCharges dtype: {final_dtype}")
print(f"Coerced missing values: {coerced_missing_count}")
print(
    "Handling method: coerced non-numeric entries to NaN, then filled missing TotalCharges with 0.0"
)
print(f"Final row count: {final_row_count}")


### 1.2: Validate the analysis population, churn group sizes, and feature availability.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

# Use the existing cleaned dataset directly

df_population = df_customer_churn_cleaned

total_customers = int(len(df_population))
churn_counts = df_population["Churn"].value_counts(dropna=False)
churned_count = int(churn_counts.get("Yes", 0))
retained_count = int(churn_counts.get("No", 0))
churned_share = churned_count / total_customers if total_customers else np.nan
retained_share = retained_count / total_customers if total_customers else np.nan

candidate_fields = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "tenure",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "MonthlyCharges",
    "TotalCharges",
]

missing_candidate_fields = [
    col for col in candidate_fields if col not in df_population.columns
]
fields_available_before_churn = len(missing_candidate_fields) == 0

# Assess leakage signal using numeric features versus churn outcome
# Convert churn to binary where Yes=1, No=0

df_leakage = df_population[
    ["Churn"]
    + [
        col
        for col in df_population.columns
        if pd.api.types.is_numeric_dtype(df_population[col])
    ]
].copy()
df_leakage["churn_binary"] = df_leakage["Churn"].map({"Yes": 1, "No": 0})

numeric_feature_cols = [
    col for col in df_leakage.columns if col not in ["Churn", "churn_binary"]
]
correlations = {}
for col in numeric_feature_cols:
    series_x = pd.to_numeric(df_leakage[col], errors="coerce")
    valid_mask = series_x.notna() & df_leakage["churn_binary"].notna()
    if valid_mask.sum() >= 2:
        corr_value = series_x[valid_mask].corr(
            df_leakage.loc[valid_mask, "churn_binary"]
        )
        if pd.notna(corr_value):
            correlations[col] = float(corr_value)

high_corr_flags = {col: corr for col, corr in correlations.items() if abs(corr) > 0.8}

print("Validation summary:")
print(f"Total customers: {total_customers}")
print(f"Churned customers: {churned_count} ({churned_share:.1%})")
print(f"Retained customers: {retained_count} ({retained_share:.1%})")

if fields_available_before_churn:
    print("Candidate modeling fields are available before churn occurs: yes")
else:
    print("Candidate modeling fields are available before churn occurs: no")
    print(f"Missing candidate fields: {', '.join(missing_candidate_fields)}")

if high_corr_flags:
    print("Potential leakage signal found:")
    for col, corr_value in sorted(
        high_corr_flags.items(), key=lambda x: abs(x[1]), reverse=True
    ):
        print(f"{col}: correlation with churn = {corr_value:.3f}")
else:
    print(
        "No numeric feature showed absolute correlation with churn above 0.8; no leakage signal was found."
    )


## Task 2: Build, evaluate, and interpret a churn prediction model, including feature importance and lift chart to assess business value.

**Acceptance Criteria:** A validated churn prediction model with proper train/test split, baseline comparison, feature importance, and a lift chart showing model value in ranking churners.

### 2.1: Split the data into stratified train/test sets and establish a naive baseline.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

# Use the cleaned churn dataset already available in the namespace

df_model_base = df_customer_churn_cleaned.copy()

# Define pre-churn predictors: exclude customer identifier and churn outcome
predictor_cols = [
    col for col in df_model_base.columns if col not in ["customerID", "Churn"]
]

# Target encoding: Yes=1, No=0
df_model_base["churn_binary"] = (
    df_model_base["Churn"].map({"Yes": 1, "No": 0}).astype(int)
)

# One-hot encode categorical predictors and keep all columns numeric for modeling
X_all = pd.get_dummies(df_model_base[predictor_cols], drop_first=True).astype(float)
y_all = df_model_base["churn_binary"].astype(int)

# Stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

# Naive baseline: predict the majority class from the training set
majority_class = int(y_train.mode().iloc[0])
y_pred_baseline = np.full(shape=len(y_test), fill_value=majority_class, dtype=int)

# For ROC-AUC, use constant probabilities equal to the training positive rate
baseline_positive_rate = float(y_train.mean())
y_prob_baseline = np.full(
    shape=len(y_test), fill_value=baseline_positive_rate, dtype=float
)

acc_value = accuracy_score(y_test, y_pred_baseline)
prec_value = precision_score(y_test, y_pred_baseline, zero_division=0)
rec_value = recall_score(y_test, y_pred_baseline, zero_division=0)
roc_auc_value = roc_auc_score(y_test, y_prob_baseline)

print("Train/test split summary:")
print(f"Train rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print(f"Predictor count after encoding: {X_all.shape[1]}")
print("Baseline benchmark on test set:")
print(f"Accuracy: {acc_value:.3f}")
print(f"Precision: {prec_value:.3f}")
print(f"Recall: {rec_value:.3f}")
print(f"ROC-AUC: {roc_auc_value:.3f}")


### 2.2: Train a predictive model for churn using only pre-churn features.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# Train a churn classification model using the existing stratified split
# Preprocessing is included inside the modeling workflow

df_model_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

df_model_pipeline.fit(X_train, y_train)

y_pred_model = df_model_pipeline.predict(X_test)
y_prob_model = df_model_pipeline.predict_proba(X_test)[:, 1]

acc_value = accuracy_score(y_test, y_pred_model)
prec_value = precision_score(y_test, y_pred_model, zero_division=0)
rec_value = recall_score(y_test, y_pred_model, zero_division=0)
f1_value = f1_score(y_test, y_pred_model, zero_division=0)
roc_auc_value = roc_auc_score(y_test, y_prob_model)

print("Churn model performance on test set:")
print(f"Accuracy: {acc_value:.3f}")
print(f"Precision: {prec_value:.3f}")
print(f"Recall: {rec_value:.3f}")
print(f"F1: {f1_value:.3f}")
print(f"ROC-AUC: {roc_auc_value:.3f}")


### 2.3: Identify the top features driving churn risk.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

# Extract the fitted logistic regression from the existing pipeline
model_step = df_model_pipeline.named_steps["classifier"]
feature_names = list(X_train.columns)
coefficients = pd.Series(model_step.coef_[0], index=feature_names)

# Build a signed importance table
# Positive coefficient => raises churn risk; negative => lowers churn risk

df_feature_influence = pd.DataFrame(
    {
        "feature": coefficients.index,
        "coefficient": coefficients.values,
        "abs_coefficient": coefficients.abs().values,
        "risk_direction": np.where(
            coefficients.values > 0, "Raises churn risk", "Lowers churn risk"
        ),
    }
)

df_feature_influence["feature_group"] = df_feature_influence["feature"].apply(
    lambda x: (
        "Commercial"
        if x in ["MonthlyCharges", "TotalCharges"]
        else (
            "Tenure"
            if x == "tenure"
            else (
                "Payment"
                if x.startswith("PaymentMethod_")
                else (
                    "Contract"
                    if x.startswith("Contract_")
                    else (
                        "Service"
                        if any(
                            x.startswith(prefix)
                            for prefix in [
                                "InternetService_",
                                "OnlineSecurity_",
                                "OnlineBackup_",
                                "DeviceProtection_",
                                "TechSupport_",
                                "StreamingTV_",
                                "StreamingMovies_",
                                "PhoneService_",
                                "MultipleLines_",
                            ]
                        )
                        else (
                            "Demographic"
                            if x
                            in [
                                "SeniorCitizen",
                                "gender_Male",
                                "Partner_Yes",
                                "Dependents_Yes",
                                "PaperlessBilling_Yes",
                            ]
                            else "Other"
                        )
                    )
                )
            )
        )
    )
)

# Rank by strongest absolute impact

df_top10_influence = (
    df_feature_influence.sort_values("abs_coefficient", ascending=False).head(10).copy()
)
df_top10_influence["coefficient"] = df_top10_influence["coefficient"].round(4)
df_top10_influence["abs_coefficient"] = df_top10_influence["abs_coefficient"].round(4)

print("Top 10 churn drivers from the logistic regression model:")
print(
    df_top10_influence[
        ["feature", "feature_group", "coefficient", "risk_direction"]
    ].to_string(index=False)
)


### 2.4: Evaluate model performance using a lift chart to assess business value.
_Output: figure_

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Build test-set lift / cumulative gains table

df_lift_chart = pd.DataFrame(
    {
        "y_true": y_test.reset_index(drop=True).astype(int),
        "y_score": np.asarray(y_prob_model, dtype=float),
    }
)

df_lift_chart = df_lift_chart.sort_values("y_score", ascending=False).reset_index(
    drop=True
)
df_lift_chart["risk_rank"] = np.arange(1, len(df_lift_chart) + 1)
df_lift_chart["decile"] = (
    pd.qcut(df_lift_chart["risk_rank"], 10, labels=False, duplicates="drop") + 1
)

# Decile summary for annotation and table completeness

df_decile_summary = (
    df_lift_chart.groupby("decile")
    .agg(customers=("y_true", "count"), churners=("y_true", "sum"))
    .reset_index()
    .sort_values("decile")
)

df_decile_summary["cum_customers"] = df_decile_summary["customers"].cumsum()
df_decile_summary["cum_churners"] = df_decile_summary["churners"].cumsum()
df_decile_summary["cum_customer_share"] = df_decile_summary["cum_customers"] / len(
    df_lift_chart
)
df_decile_summary["cum_churner_share"] = (
    df_decile_summary["cum_churners"] / df_decile_summary["churners"].sum()
)
df_decile_summary["random_baseline"] = df_decile_summary["cum_customer_share"]

# Title states the share of churners captured in the top share of risk scores
capture_top_30 = (
    float(
        df_lift_chart.head(max(1, int(np.ceil(0.30 * len(df_lift_chart)))))[
            "y_true"
        ].sum()
        / df_lift_chart["y_true"].sum()
    )
    if df_lift_chart["y_true"].sum() > 0
    else 0.0
)
chart_title = f"Top 30% of risk scores captures {capture_top_30:.1%} of churners: cumulative gains lift chart"

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df_decile_summary["cum_customer_share"],
        y=df_decile_summary["cum_churner_share"],
        mode="lines+markers",
        name="Model lift",
        line=dict(color="#1f77b4", width=3),
    )
)
fig.add_trace(
    go.Scatter(
        x=df_decile_summary["cum_customer_share"],
        y=df_decile_summary["random_baseline"],
        mode="lines",
        name="Random baseline",
        line=dict(color="gray", width=2, dash="dash"),
    )
)
fig.update_layout(
    title=chart_title,
    xaxis_title="Cumulative Customers Targeted",
    yaxis_title="Cumulative Churners Captured",
    legend_title_text="",
)
fig.update_xaxes(tickformat=".0%")
fig.update_yaxes(tickformat=".0%")
fig.show()

print("Lift chart summary:")
print(f"Test customers: {len(df_lift_chart)}")
print(f"Total churners in test set: {int(df_lift_chart['y_true'].sum())}")
print(f"Churners captured in top 30% risk scores: {capture_top_30:.1%}")


## Task 3: Segment customers by churn risk, profile high-risk groups, and estimate monthly revenue at risk to inform retention strategies.

**Acceptance Criteria:** Customers are segmented into risk tiers, high-risk profiles are characterized, and monthly revenue at risk is estimated and visualized.

### 3.1: Assign customers to churn risk tiers based on predicted probabilities.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

# Align holdout customer identifiers with the existing test split
if isinstance(X_test, pd.DataFrame):
    df_test_customers = df_model_base.loc[
        X_test.index, ["customerID", "MonthlyCharges", "Churn"]
    ].copy()
else:
    df_test_customers = df_model_base.iloc[y_test.index][
        ["customerID", "MonthlyCharges", "Churn"]
    ].copy()

# Build holdout risk profile table
n_test = len(df_test_customers)
df_risk_profiles = df_test_customers.reset_index(drop=True).copy()
df_risk_profiles["predicted_churn_probability"] = pd.Series(
    np.asarray(y_prob_model, dtype=float), index=df_risk_profiles.index
)

# Assign risk tiers based on predicted probability rank
risk_rank = df_risk_profiles["predicted_churn_probability"].rank(
    method="first", ascending=False
)
cut_high = max(1, int(np.ceil(0.10 * n_test)))
cut_medium = max(cut_high + 1, int(np.ceil(0.30 * n_test)))

df_risk_profiles["risk_tier"] = np.where(
    risk_rank <= cut_high, "High", np.where(risk_rank <= cut_medium, "Medium", "Low")
)

# Order columns as requested
df_risk_profiles = df_risk_profiles[
    [
        "customerID",
        "predicted_churn_probability",
        "risk_tier",
        "MonthlyCharges",
        "Churn",
    ]
].copy()

add_to_workspace(df_risk_profiles)

# Compact summary of customer counts by tier
print("Risk tier summary:")
print(
    df_risk_profiles["risk_tier"]
    .value_counts()
    .reindex(["High", "Medium", "Low"])
    .fillna(0)
    .astype(int)
    .to_string()
)


### 3.2: Characterize the defining features of high-risk customers.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

# Use existing saved risk-tier assignments and align them back to the holdout customer rows
if "df_test_customers" in globals():
    df_risk_base = df_test_customers.merge(
        df_risk_profiles[["customerID", "risk_tier", "predicted_churn_probability"]],
        on="customerID",
        how="left",
    ).copy()
else:
    df_risk_base = df_risk_profiles.merge(
        df_model_base[
            [
                "customerID",
                "tenure",
                "Contract",
                "PaymentMethod",
                "PhoneService",
                "MultipleLines",
                "InternetService",
                "OnlineSecurity",
                "OnlineBackup",
                "DeviceProtection",
                "TechSupport",
                "StreamingTV",
                "StreamingMovies",
            ]
        ].drop_duplicates("customerID"),
        on="customerID",
        how="left",
    ).copy()

# If df_test_customers exists, enrich with customer attributes from the full model base
if "df_test_customers" in globals():
    df_risk_base = df_risk_base.merge(
        df_model_base[
            [
                "customerID",
                "tenure",
                "Contract",
                "PaymentMethod",
                "PhoneService",
                "MultipleLines",
                "InternetService",
                "OnlineSecurity",
                "OnlineBackup",
                "DeviceProtection",
                "TechSupport",
                "StreamingTV",
                "StreamingMovies",
            ]
        ].drop_duplicates("customerID"),
        on="customerID",
        how="left",
    ).copy()

# Ensure monthly revenue and churn are present for the summary
if "MonthlyCharges" not in df_risk_base.columns:
    df_risk_base = df_risk_base.merge(
        df_test_customers[["customerID", "MonthlyCharges", "Churn"]],
        on="customerID",
        how="left",
    ).copy()

# Actual churn rate by tier
if "Churn" in df_risk_base.columns:
    df_risk_base["churn_binary"] = df_risk_base["Churn"].map({"Yes": 1, "No": 0})
else:
    df_risk_base["churn_binary"] = np.nan

df_tier_churn_rate = (
    df_risk_base.groupby("risk_tier", as_index=False)
    .agg(customers=("customerID", "count"), actual_churn_rate=("churn_binary", "mean"))
    .reindex([0, 1, 2])
)

# The reindex above is safe only if positions were preserved; rebuild cleanly if needed
if df_tier_churn_rate["risk_tier"].isna().any():
    df_tier_churn_rate = df_risk_base.groupby("risk_tier", as_index=False).agg(
        customers=("customerID", "count"), actual_churn_rate=("churn_binary", "mean")
    )

# High-risk segment profile
risk_order = ["High", "Medium", "Low"]
df_high_risk = df_risk_base[df_risk_base["risk_tier"] == "High"].copy()
df_high_risk_summary = pd.DataFrame(
    {
        "metric": [
            "Customers",
            "Median tenure (months)",
            "Average MonthlyCharges",
            "Most common Contract",
            "Most common PaymentMethod",
            "PhoneService adoption",
            "MultipleLines adoption",
            "Fiber optic adoption",
            "OnlineSecurity adoption",
            "OnlineBackup adoption",
            "DeviceProtection adoption",
            "TechSupport adoption",
            "StreamingTV adoption",
            "StreamingMovies adoption",
        ],
        "high_risk_value": [
            int(len(df_high_risk)),
            float(df_high_risk["tenure"].median()),
            float(df_high_risk["MonthlyCharges"].mean()),
            (
                df_high_risk["Contract"].mode().iloc[0]
                if not df_high_risk["Contract"].mode().empty
                else np.nan
            ),
            (
                df_high_risk["PaymentMethod"].mode().iloc[0]
                if not df_high_risk["PaymentMethod"].mode().empty
                else np.nan
            ),
            float((df_high_risk["PhoneService"] == "Yes").mean()),
            float((df_high_risk["MultipleLines"] == "Yes").mean()),
            float((df_high_risk["InternetService"] == "Fiber optic").mean()),
            float((df_high_risk["OnlineSecurity"] == "Yes").mean()),
            float((df_high_risk["OnlineBackup"] == "Yes").mean()),
            float((df_high_risk["DeviceProtection"] == "Yes").mean()),
            float((df_high_risk["TechSupport"] == "Yes").mean()),
            float((df_high_risk["StreamingTV"] == "Yes").mean()),
            float((df_high_risk["StreamingMovies"] == "Yes").mean()),
        ],
    }
)

# Compact churn rate table by tier
df_tier_churn_rate = df_tier_churn_rate[
    ["risk_tier", "customers", "actual_churn_rate"]
].copy()
df_tier_churn_rate["actual_churn_rate"] = (
    df_tier_churn_rate["actual_churn_rate"].fillna(0).round(3)
)
df_tier_churn_rate = df_tier_churn_rate.sort_values(
    "risk_tier", key=lambda s: s.map({k: i for i, k in enumerate(risk_order)})
).reset_index(drop=True)

# Format profile outputs for decision-ready printing
for col in ["high_risk_value"]:
    pass

print("High-risk segment profile:")
print(
    f"Customers: {int(df_high_risk_summary.loc[df_high_risk_summary['metric'] == 'Customers', 'high_risk_value'].iloc[0])}"
)
print(
    f"Median tenure (months): {df_high_risk_summary.loc[df_high_risk_summary['metric'] == 'Median tenure (months)', 'high_risk_value'].iloc[0]:.1f}"
)
print(
    f"Average MonthlyCharges: {df_high_risk_summary.loc[df_high_risk_summary['metric'] == 'Average MonthlyCharges', 'high_risk_value'].iloc[0]:.2f}"
)
print(
    f"Most common Contract: {df_high_risk_summary.loc[df_high_risk_summary['metric'] == 'Most common Contract', 'high_risk_value'].iloc[0]}"
)
print(
    f"Most common PaymentMethod: {df_high_risk_summary.loc[df_high_risk_summary['metric'] == 'Most common PaymentMethod', 'high_risk_value'].iloc[0]}"
)
print(
    f"PhoneService adoption: {df_high_risk_summary.loc[df_high_risk_summary['metric'] == 'PhoneService adoption', 'high_risk_value'].iloc[0]:.1%}"
)
print(
    f"MultipleLines adoption: {df_high_risk_summary.loc[df_high_risk_summary['metric'] == 'MultipleLines adoption', 'high_risk_value'].iloc[0]:.1%}"
)
print(
    f"Fiber optic adoption: {df_high_risk_summary.loc[df_high_risk_summary['metric'] == 'Fiber optic adoption', 'high_risk_value'].iloc[0]:.1%}"
)
print(
    f"OnlineSecurity adoption: {df_high_risk_summary.loc[df_high_risk_summary['metric'] == 'OnlineSecurity adoption', 'high_risk_value'].iloc[0]:.1%}"
)
print(
    f"OnlineBackup adoption: {df_high_risk_summary.loc[df_high_risk_summary['metric'] == 'OnlineBackup adoption', 'high_risk_value'].iloc[0]:.1%}"
)
print(
    f"DeviceProtection adoption: {df_high_risk_summary.loc[df_high_risk_summary['metric'] == 'DeviceProtection adoption', 'high_risk_value'].iloc[0]:.1%}"
)
print(
    f"TechSupport adoption: {df_high_risk_summary.loc[df_high_risk_summary['metric'] == 'TechSupport adoption', 'high_risk_value'].iloc[0]:.1%}"
)
print(
    f"StreamingTV adoption: {df_high_risk_summary.loc[df_high_risk_summary['metric'] == 'StreamingTV adoption', 'high_risk_value'].iloc[0]:.1%}"
)
print(
    f"StreamingMovies adoption: {df_high_risk_summary.loc[df_high_risk_summary['metric'] == 'StreamingMovies adoption', 'high_risk_value'].iloc[0]:.1%}"
)
print("")
print("Actual churn rate by risk tier:")
print(
    df_tier_churn_rate.to_string(
        index=False, formatters={"actual_churn_rate": "{:.1%}".format}
    )
)


### 3.3: Estimate the monthly revenue at risk in each churn risk segment.
_Output: figure_

In [ ]:
import pandas as pd
import plotly.express as px

# Aggregate the saved risk-profile table by tier
df_revenue_at_risk_by_tier = df_risk_profiles.groupby("risk_tier", as_index=False).agg(
    customers=("customerID", "count"),
    total_monthly_revenue=("MonthlyCharges", "sum"),
    average_monthly_revenue=("MonthlyCharges", "mean"),
    actual_churn_rate=("Churn", lambda s: (s == "Yes").mean()),
)

risk_tier_order = ["High", "Medium", "Low"]
df_revenue_at_risk_by_tier["risk_tier"] = pd.Categorical(
    df_revenue_at_risk_by_tier["risk_tier"], categories=risk_tier_order, ordered=True
)
df_revenue_at_risk_by_tier = df_revenue_at_risk_by_tier.sort_values(
    "risk_tier"
).reset_index(drop=True)

df_revenue_at_risk_by_tier["total_monthly_revenue"] = df_revenue_at_risk_by_tier[
    "total_monthly_revenue"
].round(2)
df_revenue_at_risk_by_tier["average_monthly_revenue"] = df_revenue_at_risk_by_tier[
    "average_monthly_revenue"
].round(2)
df_revenue_at_risk_by_tier["actual_churn_rate"] = df_revenue_at_risk_by_tier[
    "actual_churn_rate"
].round(3)

add_to_workspace(df_revenue_at_risk_by_tier)

print("Revenue at risk by tier:")
print(
    df_revenue_at_risk_by_tier.to_string(
        index=False,
        formatters={
            "actual_churn_rate": "{:.1%}".format,
            "total_monthly_revenue": "{:,.2f}".format,
            "average_monthly_revenue": "{:,.2f}".format,
        },
    )
)

fig = px.bar(
    df_revenue_at_risk_by_tier,
    x="risk_tier",
    y="total_monthly_revenue",
    color="risk_tier",
    category_orders={"risk_tier": risk_tier_order},
    title="Revenue at Risk Is Concentrated in the High-Risk Customer Tier",
    labels={
        "risk_tier": "Risk Tier",
        "total_monthly_revenue": "Total Monthly Revenue (USD)",
    },
    color_discrete_map={"High": "#d62728", "Medium": "#ff7f0e", "Low": "#1f77b4"},
)
fig.update_traces(marker_line_color="black", marker_line_width=1)
fig.update_yaxes(tickformat=",")
fig.show()
